# ACS API Request

In [1]:
# Import libraries
import pandas as pd
import requests
import time

In [2]:
# Geographic codes
relfile = pd.read_csv(r'C:\Users\HerreraMartinezR\OneDrive - NYC Office of Management and Budget\HTM Projects\ACS_Warehouse\docs\tab20_puma520_cousub20_natl.txt', sep = '|', header = 0, dtype = 'str')
relfile.head()

,OID_PUMA5_20,GEOID_PUMA5_20,NAMELSAD_PUMA5_20,AREALAND_PUMA5_20,AREAWATER_PUMA5_20,MTFCC_PUMA5_20,FUNCSTAT_PUMA5_20,OID_COUSUB_20,GEOID_COUSUB_20,NAMELSAD_COUSUB_20,AREALAND_COUSUB_20,AREAWATER_COUSUB_20,MTFCC_COUSUB_20,CLASSFP_COUSUB_20,FUNCSTAT_COUSUB_20,AREALAND_PART,AREAWATER_PART
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27690590833492,6001037700,Ituau county,12986619,51344770,G4040,T1,A,12986619,51344770
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27690590833493,6001051300,Ma'oputasi county,17111108,30945042,G4040,T1,A,17111108,30945042
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27690590833494,6001067300,Sa'ole county,5954576,47875714,G4040,T1,A,5954576,47875714
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27690590833495,6001072100,Sua county,17200142,63697222,G4040,T1,A,17200142,63697222
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27690590833496,6001086500,Vaifanua county,11872421,155861930,G4040,T1,A,11872421,155861930


In [3]:
# Extract PUMA codes for NYC
pumas_nyc = relfile.loc[relfile['NAMELSAD_PUMA5_20'].str.contains('NYC') == True, 'GEOID_PUMA5_20'].reset_index(drop = True)
# Drop the state id
pumas_nyc = pumas_nyc.str[2:7].copy()
# Drop missing values
pumas_nyc = pumas_nyc.dropna()
# Drop any duplicates
pumas_nyc = pumas_nyc.drop_duplicates().reset_index(drop = True)
# Check the results
pumas_nyc.head()

0    04103
1    04104
2    04107
3    04108
4    04109
Name: GEOID_PUMA5_20, dtype: object

In [4]:
# API  Key 
with open(r"C:\Users\HerreraMartinezR\OneDrive - NYC Office of Management and Budget\Data\API_Keys\CensusAPIKey.txt", 'r') as file:
   MyKey = file.read().strip()

## DP02

In [5]:
# API Key and URL
api_key = MyKey

# API for ACS 1-year Data Profile Tables
api_url_2024 = 'https://api.census.gov/data/2024/acs/acs1/profile'
api_url_2023 = 'https://api.census.gov/data/2023/acs/acs1/profile'
api_url_2022 = 'https://api.census.gov/data/2022/acs/acs1/profile'
# api_url_2021 = 'https://api.census.gov/data/2021/acs/acs1/profile' # NOTE: Limited number of pumas available
# api_url_2020 = 'https://api.census.gov/data/2020/acs/acs1/profile' # NOTE: Not released due to pandemic

# Tables/variables
variables =  ['group(DP02)']  

In [6]:
# Request
   
geo_code =  pumas_nyc.tolist()      
api_list = [api_url_2024, api_url_2023, api_url_2022]
vintages = [2024, 2023, 2022]

dfs_by_vintage = {}

for url, year in zip(api_list, vintages):
    rows = []
    for code in geo_code:
        params = {
            "get": ",".join(variables),
            "for": f"public use microdata area:{code}", 
            "in": "state:36",
            "key": api_key
        }
        try:
            resp = requests.get(url, params=params, timeout=60)
            resp.raise_for_status()
            json_data = resp.json()

            headers = json_data[0]
            for vals in json_data[1:]:
                rec = dict(zip(headers, vals))
                rec["vintage"] = year
                rec['sample'] = url.split("/acs/")[1].split("/")[0]     # TEST: Including ACS sample type in the output (1 year or 5 year)
                rec['group'] = variables[0]                             # TEST: Include the variable group. WILL BE AN ISSUE FOR IND. VAR??
                rows.append(rec)

        except requests.RequestException as e:
            print(f"[{year}] Error fetching puma {code}: {e}")

        time.sleep(0.15)                                            # be nice to the API

    df_year = pd.DataFrame(rows)
    
    # Optional: make numeric columns numeric
    non_numeric = {"place", "state", "county", "GEO_ID" , "NAME", "vintage", "sample", "group"}
    num_cols = [c for c in df_year.columns if c not in non_numeric]
    df_year[num_cols] = df_year[num_cols].apply(pd.to_numeric, errors="coerce")

    dfs_by_vintage[year] = df_year

In [7]:
# Store as df
df_2022 = dfs_by_vintage[2022]
df_2023 = dfs_by_vintage[2023]
df_2024 = dfs_by_vintage[2024]

print('2022 df shape', df_2022.shape)
print('2023 df shape', df_2023.shape)
print('2024 df shape', df_2024.shape)

2022 df shape (55, 1239)
2023 df shape (55, 1239)
2024 df shape (55, 1239)


In [8]:
df_2022

,DP02_0001E,DP02_0001EA,DP02_0001M,DP02_0001MA,DP02_0001PE,DP02_0001PEA,DP02_0001PM,DP02_0001PMA,DP02_0002E,DP02_0002EA,...,DP02_0154PEA,DP02_0154PM,DP02_0154PMA,GEO_ID,NAME,state,public use microdata area,vintage,sample,group
0,73474,NaN,4448,NaN,73474,NaN,-888888888,NaN,15232,NaN,...,NaN,3.3,NaN,795P200US3604103,NYC-Manhattan Community District 3--Lower East...,36,4103,2022,acs1,group(DP02)
1,67653,NaN,4580,NaN,67653,NaN,-888888888,NaN,13438,NaN,...,NaN,3.1,NaN,795P200US3604104,NYC-Manhattan Community District 4--Chelsea & ...,36,4104,2022,acs1,group(DP02)
2,115928,NaN,6894,NaN,115928,NaN,-888888888,NaN,42508,NaN,...,NaN,2.2,NaN,795P200US3604107,NYC-Manhattan Community District 7--Upper West...,36,4107,2022,acs1,group(DP02)
3,114602,NaN,5481,NaN,114602,NaN,-888888888,NaN,41410,NaN,...,NaN,1.4,NaN,795P200US3604108,NYC-Manhattan Community District 8--Upper East...,36,4108,2022,acs1,group(DP02)
4,44758,NaN,3670,NaN,44758,NaN,-888888888,NaN,9119,NaN,...,NaN,4.5,NaN,795P200US3604109,NYC-Manhattan Community District 9--Morningsid...,36,4109,2022,acs1,group(DP02)
5,63590,NaN,4485,NaN,63590,NaN,-888888888,NaN,12530,NaN,...,NaN,3.7,NaN,795P200US3604110,NYC-Manhattan Community District 10--Harlem PU...,36,4110,2022,acs1,group(DP02)
6,57494,NaN,3883,NaN,57494,NaN,-888888888,NaN,11026,NaN,...,NaN,4.1,NaN,795P200US3604111,NYC-Manhattan Community District 11--East Harl...,36,4111,2022,acs1,group(DP02)
7,82694,NaN,4856,NaN,82694,NaN,-888888888,NaN,22416,NaN,...,NaN,2.8,NaN,795P200US3604112,NYC-Manhattan Community District 12--Washingto...,36,4112,2022,acs1,group(DP02)
8,78584,NaN,5257,NaN,78584,NaN,-888888888,NaN,22643,NaN,...,NaN,1.9,NaN,795P200US3604121,NYC-Manhattan Community Districts 1 & 2--Finan...,36,4121,2022,acs1,group(DP02)
9,105067,NaN,6067,NaN,105067,NaN,-888888888,NaN,27453,NaN,...,NaN,1.7,NaN,795P200US3604165,NYC-Manhattan Community Districts 5 & 6--Midto...,36,4165,2022,acs1,group(DP02)


In [9]:
# Save
df_2022.to_csv(r"C:\Users\HerreraMartinezR\OneDrive - NYC Office of Management and Budget\Data\ACS\1_yr_sample\2022\PUMA\1_yr_ACS_2022_PUMA_NYC_DP02.csv", index = False)
df_2023.to_csv(r"C:\Users\HerreraMartinezR\OneDrive - NYC Office of Management and Budget\Data\ACS\1_yr_sample\2023\PUMA\1_yr_ACS_2023_PUMA_NYC_DP02.csv", index = False)
df_2024.to_csv(r"C:\Users\HerreraMartinezR\OneDrive - NYC Office of Management and Budget\Data\ACS\1_yr_sample\2024\PUMA\1_yr_ACS_2024_PUMA_NYC_DP02.csv", index = False)

## DP03

In [10]:
# API Key and URL
api_key = MyKey

# API for ACS 1-year Data Profile Tables
api_url_2024 = 'https://api.census.gov/data/2024/acs/acs1/profile'
api_url_2023 = 'https://api.census.gov/data/2023/acs/acs1/profile'
api_url_2022 = 'https://api.census.gov/data/2022/acs/acs1/profile'
# api_url_2021 = 'https://api.census.gov/data/2021/acs/acs1/profile' # NOTE: Limited number of pumas available
# api_url_2020 = 'https://api.census.gov/data/2020/acs/acs1/profile' # NOTE: Not released due to pandemic

# Tables/variables
variables =  ['group(DP03)']  

In [ ]:
# Request
   
geo_code =  pumas_nyc.tolist()      
api_list = [api_url_2024, api_url_2023, api_url_2022]
vintages = [2024, 2023, 2022]
# geo_code = ['04313', '04316', '04317']      # For tests
# api_list = [api_url_2024]                   # For tests
# vintages = [2024]                           # For tests

dfs_by_vintage = {}

for url, year in zip(api_list, vintages):
    rows = []
    for code in geo_code:
        params = {
            "get": ",".join(variables),
            "for": f"public use microdata area:{code}", 
            "in": "state:36",
            "key": api_key
        }
        try:
            resp = requests.get(url, params=params, timeout=90)
            resp.raise_for_status()
            json_data = resp.json()

            headers = json_data[0]
            for vals in json_data[1:]:
                rec = dict(zip(headers, vals))
                rec["vintage"] = year
                rec['sample'] = url.split("/acs/")[1].split("/")[0]     # TEST: Including ACS sample type in the output (1 year or 5 year)
                rec['group'] = variables[0]                             # TEST: Include the variable group. WILL BE AN ISSUE FOR IND. VAR??                
                rows.append(rec)

        except requests.RequestException as e:
            print(f"[{year}] Error fetching puma {code}: {e}")

        time.sleep(0.15)                                            # be nice to the API

    df_year = pd.DataFrame(rows)
    
    # Optional: make numeric columns numeric
    non_numeric = {"place", "state", "county", "GEO_ID" , "NAME", "vintage", "sample", "group"}
    num_cols = [c for c in df_year.columns if c not in non_numeric]
    df_year[num_cols] = df_year[num_cols].apply(pd.to_numeric, errors="coerce")

    dfs_by_vintage[year] = df_year

In [12]:
# Store as df
df_2022 = dfs_by_vintage[2022]
df_2023 = dfs_by_vintage[2023]
df_2024 = dfs_by_vintage[2024]

print('2022 df shape', df_2022.shape)
print('2023 df shape', df_2023.shape)
print('2024 df shape', df_2024.shape)

2022 df shape (55, 1103)
2023 df shape (55, 1103)
2024 df shape (55, 1103)


In [13]:
# Save
df_2022.to_csv(r"C:\Users\HerreraMartinezR\OneDrive - NYC Office of Management and Budget\Data\ACS\1_yr_sample\2022\PUMA\1_yr_ACS_2022_PUMA_NYC_DP03.csv", index = False)
df_2023.to_csv(r"C:\Users\HerreraMartinezR\OneDrive - NYC Office of Management and Budget\Data\ACS\1_yr_sample\2023\PUMA\1_yr_ACS_2023_PUMA_NYC_DP03.csv", index = False)
df_2024.to_csv(r"C:\Users\HerreraMartinezR\OneDrive - NYC Office of Management and Budget\Data\ACS\1_yr_sample\2024\PUMA\1_yr_ACS_2024_PUMA_NYC_DP03.csv", index = False)

## DP04

In [15]:
# API Key and URL
api_key = MyKey

# API for ACS 1-year Data Profile Tables
api_url_2024 = 'https://api.census.gov/data/2024/acs/acs1/profile'
api_url_2023 = 'https://api.census.gov/data/2023/acs/acs1/profile'
api_url_2022 = 'https://api.census.gov/data/2022/acs/acs1/profile'
# api_url_2021 = 'https://api.census.gov/data/2021/acs/acs1/profile' # NOTE: Limited number of pumas available
# api_url_2020 = 'https://api.census.gov/data/2020/acs/acs1/profile' # NOTE: Not released due to pandemic

# Tables/variables
variables =  ['group(DP04)']  

In [16]:
# Request
   
geo_code =  pumas_nyc.tolist()      
api_list = [api_url_2024, api_url_2023, api_url_2022]
vintages = [2024, 2023, 2022]
# geo_code = ['04313', '04316', '04317']      # For tests
# api_list = [api_url_2024]                   # For tests
# vintages = [2024]                           # For tests

dfs_by_vintage = {}

for url, year in zip(api_list, vintages):
    rows = []
    for code in geo_code:
        params = {
            "get": ",".join(variables),
            "for": f"public use microdata area:{code}", 
            "in": "state:36",
            "key": api_key
        }
        try:
            resp = requests.get(url, params=params, timeout=90)
            resp.raise_for_status()
            json_data = resp.json()

            headers = json_data[0]
            for vals in json_data[1:]:
                rec = dict(zip(headers, vals))
                rec["vintage"] = year
                rec['sample'] = url.split("/acs/")[1].split("/")[0]     # TEST: Including ACS sample type in the output (1 year or 5 year)
                rec['group'] = variables[0]                             # TEST: Include the variable group. WILL BE AN ISSUE FOR IND. VAR??
                rows.append(rec)

        except requests.RequestException as e:
            print(f"[{year}] Error fetching puma {code}: {e}")

        time.sleep(0.15)                                            # be nice to the API

    df_year = pd.DataFrame(rows)
    
    # Optional: make numeric columns numeric
    non_numeric = {"place", "state", "county", "GEO_ID" , "NAME", "vintage", "sample", "group"}
    num_cols = [c for c in df_year.columns if c not in non_numeric]
    df_year[num_cols] = df_year[num_cols].apply(pd.to_numeric, errors="coerce")

    dfs_by_vintage[year] = df_year

In [17]:
# Store as df
df_2022 = dfs_by_vintage[2022]
df_2023 = dfs_by_vintage[2023]
df_2024 = dfs_by_vintage[2024]

print('2022 df shape', df_2022.shape)
print('2023 df shape', df_2023.shape)
print('2024 df shape', df_2024.shape)

2022 df shape (55, 1151)
2023 df shape (55, 1151)
2024 df shape (55, 1151)


In [18]:
# Save
df_2022.to_csv(r"C:\Users\HerreraMartinezR\OneDrive - NYC Office of Management and Budget\Data\ACS\1_yr_sample\2022\PUMA\1_yr_ACS_2022_PUMA_NYC_DP04.csv", index = False)
df_2023.to_csv(r"C:\Users\HerreraMartinezR\OneDrive - NYC Office of Management and Budget\Data\ACS\1_yr_sample\2023\PUMA\1_yr_ACS_2023_PUMA_NYC_DP04.csv", index = False)
df_2024.to_csv(r"C:\Users\HerreraMartinezR\OneDrive - NYC Office of Management and Budget\Data\ACS\1_yr_sample\2024\PUMA\1_yr_ACS_2024_PUMA_NYC_DP04.csv", index = False)

## DP05

In [19]:
# API Key and URL
api_key = MyKey

# API for ACS 1-year Data Profile Tables
api_url_2024 = 'https://api.census.gov/data/2024/acs/acs1/profile'
api_url_2023 = 'https://api.census.gov/data/2023/acs/acs1/profile'
api_url_2022 = 'https://api.census.gov/data/2022/acs/acs1/profile'
# api_url_2021 = 'https://api.census.gov/data/2021/acs/acs1/profile' # NOTE: Limited number of pumas available
# api_url_2020 = 'https://api.census.gov/data/2020/acs/acs1/profile' # NOTE: Not released due to pandemic

# Tables/variables
variables =  ['group(DP05)']  

In [20]:
# Request
   
geo_code =  pumas_nyc.tolist()      
api_list = [api_url_2024, api_url_2023, api_url_2022]
vintages = [2024, 2023, 2022]
# geo_code = ['04313', '04316', '04317']      # For tests
# api_list = [api_url_2024]                   # For tests
# vintages = [2024]                           # For tests

dfs_by_vintage = {}

for url, year in zip(api_list, vintages):
    rows = []
    for code in geo_code:
        params = {
            "get": ",".join(variables),
            "for": f"public use microdata area:{code}", 
            "in": "state:36",
            "key": api_key
        }
        try:
            resp = requests.get(url, params=params, timeout=90)
            resp.raise_for_status()
            json_data = resp.json()

            headers = json_data[0]
            for vals in json_data[1:]:
                rec = dict(zip(headers, vals))
                rec["vintage"] = year
                rec['sample'] = url.split("/acs/")[1].split("/")[0]     # TEST: Including ACS sample type in the output (1 year or 5 year)
                rec['group'] = variables[0]                             # TEST: Include the variable group. WILL BE AN ISSUE FOR IND. VAR??
                rows.append(rec)

        except requests.RequestException as e:
            print(f"[{year}] Error fetching puma {code}: {e}")

        time.sleep(0.15)                                            # be nice to the API

    df_year = pd.DataFrame(rows)
    
    # Optional: make numeric columns numeric
    non_numeric = {"place", "state", "county", "GEO_ID" , "NAME", "vintage", "sample", "group"}
    num_cols = [c for c in df_year.columns if c not in non_numeric]
    df_year[num_cols] = df_year[num_cols].apply(pd.to_numeric, errors="coerce")

    dfs_by_vintage[year] = df_year

In [21]:
# Store as df
df_2022 = dfs_by_vintage[2022]
df_2023 = dfs_by_vintage[2023]
df_2024 = dfs_by_vintage[2024]

print('2022 df shape', df_2022.shape)
print('2023 df shape', df_2023.shape)
print('2024 df shape', df_2024.shape)

2022 df shape (55, 735)
2023 df shape (55, 759)
2024 df shape (55, 871)


In [22]:
# Save
df_2022.to_csv(r"C:\Users\HerreraMartinezR\OneDrive - NYC Office of Management and Budget\Data\ACS\1_yr_sample\2022\PUMA\1_yr_ACS_2022_PUMA_NYC_DP05.csv", index = False)
df_2023.to_csv(r"C:\Users\HerreraMartinezR\OneDrive - NYC Office of Management and Budget\Data\ACS\1_yr_sample\2023\PUMA\1_yr_ACS_2023_PUMA_NYC_DP05.csv", index = False)
df_2024.to_csv(r"C:\Users\HerreraMartinezR\OneDrive - NYC Office of Management and Budget\Data\ACS\1_yr_sample\2024\PUMA\1_yr_ACS_2024_PUMA_NYC_DP05.csv", index = False)